In [ ]:
import numpy as np
import pandas as pd
import os, ast, shutil, copy, random
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from bokeh.plotting import figure, gridplot
from bokeh.io import output_file, show, output_notebook

# Configuration
warnings.filterwarnings('ignore')
output_notebook()
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.5f}'.format)

# --- SECTION 1: DATA LOADING & PREPROCESSING ---

# Load competition data
df_train_competition = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")
df_orig = pd.read_csv("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv")

# Clean original data to match competition format
df_orig = df_orig.reset_index().rename(columns={'index':'id'})
df_train_all = pd.concat([df_train_competition, df_orig]).reset_index(drop=True)
df_train = df_train_competition.copy()

LABEL = "Heart Disease"
FEATURES = [c for c in df_train.columns if c not in [LABEL, "id"]]

# Identify feature types based on cardinality
df_train_nunique = df_train[FEATURES].nunique()
NUM_FEATURES = df_train_nunique[df_train_nunique >= 50].index.tolist()
CAT_FEATURES = df_train_nunique[df_train_nunique < 50].index.tolist()

# --- SECTION 2: EXPLORATORY DATA ANALYSIS (EDA) FUNCTIONS ---

def display_two_numeric_features(col1, col2, df):
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=col1, y=col2, alpha=0.5)
    plt.title(f"Scatter: {col1} vs {col2}")
    plt.show()

def display_two_categorical_features(col1, col2, df):
    plt.figure(figsize=(10, 6))
    mx = pd.crosstab(df[col1], df[col2])
    sns.heatmap(mx, annot=True, fmt='d', cmap='YlGnBu')
    plt.title(f"Crosstab: {col1} vs {col2}")
    plt.show()

def display_three_categorical_features(col1, col2, col3, df):
    plt.figure(figsize=(12, 7))
    mx = pd.crosstab(df[col1], [df[col2], df[col3]])
    sns.heatmap(mx, annot=True, fmt='d', cmap='YlGnBu')
    plt.title(f"Crosstab: {col1} vs [{col2} & {col3}]")
    plt.show()

# Run Initial Visualizations
print("### Feature Distribution Comparison ###")
fig, axs = plt.subplots(nrows=len(FEATURES), ncols=3, figsize=(12, 3 * len(FEATURES)), layout="constrained")
for i, column in enumerate(FEATURES):
    axs[i, 0].set_title('Train: ' + column); axs[i, 0].hist(df_train[column].dropna(), color='skyblue')
    axs[i, 1].set_title('Test: ' + column); axs[i, 1].hist(df_test[column].dropna(), color='salmon')
    axs[i, 2].set_title('Original: ' + column); axs[i, 2].hist(df_orig[column].dropna(), color='green')
plt.show()

# --- SECTION 3: ENSEMBLE UTILITIES (H-BLEND) ---

def rank_normalize(s):
    r = s.rank(method="average")
    return (r - 1) / (len(r) - 1)

def sharpen(p, eps=1e-6, gamma=1.02):
    p = np.clip(p, eps, 1 - eps)
    logit = np.log(p / (1 - p))
    return 1 / (1 + np.exp(-gamma * logit))

def arr_colors(color):
    dskb, mvr = 'deepskyblue', 'mediumvioletred'
    sg = ['darkgray', 'silver', 'gainsboro']
    if color in ['red', 'R', 'Red', 'r']: return ['firebrick', 'red', 'crimson', 'tomato'] + sg
    if color in ['Green', 'G']: return ['darkgreen', 'limegreen', 'green', 'lime'] + sg
    if color in ['Blue', 'B']: return ['midnightblue', 'blue', 'mediumblue', dskb] + sg
    if color in ['RGB', 'S']: return ['mediumblue', 'darkgreen', 'crimson'] + sg
    if color in ['RGBM', 'M']: return [mvr, 'darkorchid', 'darkmagenta', 'magenta'] + sg
    return ['black', 'dimgray', 'gray'] + sg

# --- SECTION 4: VISUALIZATION FUNCTIONS FOR BLENDING ---

def seaborn_display_1(params, df_cross, show_fig1, show_fig2, color_cross):
    colors = [subm['color'] for subm in params['subm']]
    alls_df = pd.read_csv('tida_desc.csv')
    matrix = [ast.literal_eval(str(row.alls)) for row in alls_df.itertuples()]
    subms = sorted(matrix[0])
    
    # Simple correlation heatmap of submissions
    plt.figure(figsize=(10, 8))
    # Extract only submission columns for correlation
    subm_names = [s['name'] for s in params['subm']]
    # Note: This assumes tida_desc.csv contains these names which it will after h_blend runs
    # For now, we use a standard grid for visual summary
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Correlation Heatmap
    sns.heatmap(df_cross.corr(), annot=True, cmap='RdBu_r', ax=axes[0])
    axes[0].set_title("Blend Component Correlations")
    
    # KDE of final ensemble
    sns.kdeplot(df_cross[params['id_target'][1]], color=color_cross, fill=True, ax=axes[1])
    axes[1].set_title("Final Ensemble Probability Density")
    plt.show()

def matrix_vs(path, fs_names):
    dfs = [pd.read_csv(path + name + '.csv').rename(columns={"Heart Disease": name}) for name in fs_names]
    dfsm = dfs[0]
    for i in range(1, len(dfs)):
        dfsm = pd.merge(dfsm, dfs[i], on="id")
    
    m1 = pd.DataFrame({'subm': fs_names})
    dist_matrix = []
    for name_a in fs_names:
        row = []
        for name_b in fs_names:
            dist = abs(dfsm[name_a] - dfsm[name_b]).sum()
            row.append(round(dist, 2))
        dist_matrix.append(row)
    
    m2 = pd.DataFrame(dist_matrix, columns=fs_names)
    return pd.concat([m1, m2], axis=1)

# --- SECTION 5: CORE H-BLEND LOGIC ---

def h_blend(params, _update={}, cross='silver', details=False, subm=''):
    if 'path' in _update: params.update(_update)
    dk = copy.deepcopy(params)
    color_cross = cross
    
    target_col = dk['id_target'][1]
    id_col = dk['id_target'][0]
    
    # Read and merge individual submissions
    dfs_list = []
    for s in dk['subm']:
        tmp = pd.read_csv(dk['path'] + s['name'] + ".csv")
        tmp = tmp.rename(columns={target_col: s['name']})
        dfs_list.append(tmp)
        
    df_subms = dfs_list[0]
    for i in range(1, len(dfs_list)):
        df_subms = pd.merge(df_subms, dfs_list[i], on=id_col)
    
    sub_names = [s['name'] for s in dk['subm']]
    
    # Calculate rank-based ensemble
    df_subms['desc_blend'] = 0.0
    df_subms['asc_blend'] = 0.0
    
    # Sorting logic for tida_desc.csv generation
    df_subms['alls'] = df_subms[sub_names].apply(lambda x: list(x.sort_values(ascending=False).index), axis=1)
    df_subms[['id', 'alls']].to_csv('tida_desc.csv', index=False)

    # Weighted Rank Blending
    for i, s in enumerate(dk['subm']):
        weight = s['weight']
        df_subms['desc_blend'] += weight * rank_normalize(df_subms[s['name']])
        df_subms['asc_blend'] += weight * rank_normalize(df_subms[s['name']]) # Simplified for example
        
    # Final Weighted Combo
    df_subms[target_col] = (dk['type_sort'][1] * rank_normalize(df_subms['asc_blend']) + 
                            dk['type_sort'][2] * rank_normalize(df_subms['desc_blend']))
    
    # Apply Sharpening
    df_subms[target_col] = sharpen(df_subms[target_col])
    
    # Deterministic tie-break
    rng = np.random.default_rng(42)
    df_subms[target_col] += rng.normal(0, 1e-9, size=len(df_subms))
    
    if subm:
        df_subms[[id_col, target_col]].to_csv(subm, index=False)
    
    # Distances
    dist_df = matrix_vs(dk['path'], sub_names)
    print("### Submission Distances (L1) ###")
    display(dist_df)
    
    return df_subms[[id_col, target_col]]

# --- SECTION 6: EXECUTION ---

# 1. Create temporary variants for blending (Simulation of different model outputs)
# We use the provided 'Predicting Heart Disease Predict398.csv' as a base
base_df = pd.read_csv('/kaggle/input/datasets/kami1976/predictingheart-diseasedataset/Predicting Heart Disease Predict398.csv')
a = pd.read_csv('/kaggle/input/datasets/kami1976/predictingheart-diseasedataset/Predicting Heart Disease Predict397.csv')
b = pd.read_csv('/kaggle/input/datasets/kami1976/predictingheart-diseasedataset/Predicting Heart Disease Predict392.csv')
c = pd.read_csv('/kaggle/input/datasets/kami1976/predictingheart-diseasedataset/Predicting Heart Disease Predict389.csv')
d = pd.read_csv('/kaggle/input/datasets/kami1976/predictingheart-diseasedataset/Predicting Heart Disease Predict387.csv')

# Generate input files for the blender
for name, df_var in zip(['1a', '1b', '1c', '1d'], [a, b, c, d]):
    tmp = base_df.copy()
    tmp[LABEL] = tmp[LABEL] * 0.95 + 0.05 * df_var[LABEL]
    tmp.to_csv(f'{name}.csv', index=False)

# 2. Configure Blending Parameters
params = {
      'path'     : './',
      'id_target': ['id', LABEL],          
      'type_sort': ['asc/desc', 0.70, 0.30],
      'subwts'   : [-0.025, -0.025, +0.025, +0.025],
      'subm'     : [ 
          {'name': '1a', 'weight': 0.45, 'color': 'navy'},
          {'name': '1b', 'weight': 0.30, 'color': 'orange'},
          {'name': '1c', 'weight': 0.12, 'color': 'green'},
          {'name': '1d', 'weight': 0.05, 'color': 'crimson'},]
}

# 3. Run Ensemble
final_df = h_blend(params, details=True, subm='submission.csv')

# 4. Cleanup and Display
for f in ['1a', '1b', '1c', '1d', 'tida_desc']:
    if os.path.exists(f + '.csv'): os.remove(f + '.csv')

print("### Final Submission Preview ###")
display(final_df.head(15))